In [ ]:
import tensorflow_datasets as tfds

def load_wikitext103(split="train"):
    ds = tfds.load("wikitext", split=f"{split}")
    texts = []
    for example in tfds.as_numpy(ds):
        text = example["text"].decode("utf-8")
        if len(text.strip()) > 0:
            texts.append(text.strip())
    return texts

In [ ]:
pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

train_texts = dataset["train"]["text"]
val_texts   = dataset["validation"]["text"]
test_texts  = dataset["test"]["text"]

print(len(train_texts))

In [ ]:
import re

def clean_texts(texts):
    cleaned = []
    allowed_pattern = r"[^A-Za-z'.,]"
    for text in texts:
        cleaned_text = re.sub(allowed_pattern, " ", text)
        cleaned_text = re.sub(r"\s+", " ", cleaned_text)
        cleaned_text = cleaned_text.strip()
        if len(cleaned_text) > 3:
            cleaned.append(cleaned_text)
    return cleaned

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
def create_token_mappings(texts):

    all_chars = set(''.join(texts))
    alphabet = sorted(list(all_chars))
    special_tokens = ['<pad>', '<bos>', '<eos>', '<unk>']
    idx_to_char = {i: token for i, token in enumerate(special_tokens)}
    idx_to_char.update({i+4: c for i, c in enumerate(alphabet)})
    char_to_idx = {v: k for k, v in idx_to_char.items()}
    vocab_size = len(special_tokens) + len(alphabet)

    return char_to_idx, idx_to_char, vocab_size

In [ ]:
def encode_text(
    text,
    char_to_idx,
    max_len=20,
    bos_token=1,
    eos_token=2,
    pad_token=0,
    unk_token=3
):
    indices = [char_to_idx.get(c, unk_token) for c in text]
    sequence = [bos_token] + indices + [eos_token]
    if len(sequence) > max_len:
        sequence = sequence[:max_len-1] + [eos_token]
    padded = sequence + [pad_token] * (max_len - len(sequence))
    return np.array(padded, dtype=np.int32)

In [ ]:
def decode_tokens(tokens, idx_to_char, is_decoder_input=False, is_target=False):
    text = ''

    for token in tokens:
        token = int(token)

        if token == 0:
            continue

        if is_decoder_input:

            if token == 1:
                text += '<bos>'
                continue

        if is_target:

            if token == 2:
                text += '<eos>'
                break

        char = idx_to_char.get(token, f'?{token}?')

        if is_decoder_input and char in ['<eos>', '<unk>']:
            continue

        text += char
    return text

In [ ]:
texts = clean_texts(train_texts)

In [ ]:
char_to_idx, idx_to_char, vocab_size = create_token_mappings(texts)

In [ ]:
def prepare_lm_dataset(
    texts,
    char_to_idx,
    max_len=20,
    batch_size=64,
    shuffle=True,
):
    sequences = []

    for text in texts:

        encoded = encode_text(
            text,
            char_to_idx,
            max_len=max_len,
            bos_token=char_to_idx['<bos>'],
            eos_token=char_to_idx['<eos>'],
            pad_token=char_to_idx['<pad>'],
            unk_token=char_to_idx['<unk>']
        )
        sequences.append(encoded)

    sequences = np.array(sequences)
    decoder_input = sequences[:, :-1]
    target = sequences[:, 1:]

    dataset = tf.data.Dataset.from_tensor_slices(
        (decoder_input, target)
    )

    if shuffle:
        dataset = dataset.shuffle(10000)

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
lm_dataset = prepare_lm_dataset(
    texts,
    char_to_idx,
    max_len=20,
    batch_size=64
)

In [ ]:
import tensorflow as tf
from tensorflow import keras

class PerplexityMetric(keras.metrics.Metric):
    def __init__(self, name='perplexity', **kwargs):
        super().__init__(name=name, **kwargs)
        self.total_loss = self.add_weight(name='total_loss', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
        self.total_loss.assign_add(tf.reduce_sum(loss))
        self.count.assign_add(tf.cast(tf.size(loss), tf.float32))

    def result(self):
        mean_loss = self.total_loss / self.count
        return tf.exp(mean_loss)

    def reset_state(self):
        self.total_loss.assign(0.0)
        self.count.assign(0.0)

In [ ]:
import tensorflow as tf
import keras_nlp
from tensorflow import keras
from tensorflow.keras import layers

def create_char_lm_decoder(
    vocab_size,
    max_len=19,
    embedding_dim=256,
    num_heads=4,
    intermediate_dim=1024,
    num_layers=1,
    use_cross_attention=False
):
    if use_cross_attention:
        decoder_inputs = keras.Input(shape=(max_len,), dtype=tf.int32, name="decoder_input_tokens")
        encoder_inputs = keras.Input(shape=(None, embedding_dim), name="encoder_sequence")
        x = keras_nlp.layers.TokenAndPositionEmbedding(
            vocabulary_size=vocab_size,
            sequence_length=max_len,
            embedding_dim=embedding_dim,
            name="token_position_embedding"
        )(decoder_inputs)
        for i in range(num_layers):
            x = keras_nlp.layers.TransformerDecoder(
                intermediate_dim=intermediate_dim,
                num_heads=num_heads,
                dropout=0.1,
                activation="gelu",
                normalize_first=True,
                name=f"decoder_layer_{i+1}"
            )(
                decoder_sequence=x,
                encoder_sequence=encoder_inputs,
            )
        outputs = layers.Dense(vocab_size, name="logits")(x)
        model = keras.Model(
            inputs=[decoder_inputs, encoder_inputs],
            outputs=outputs,
            name="char_lm_decoder_cross_attention"
        )
    else:
        inputs = keras.Input(shape=(max_len,), dtype=tf.int32, name="decoder_input_tokens")
        x = keras_nlp.layers.TokenAndPositionEmbedding(
            vocabulary_size=vocab_size,
            sequence_length=max_len,
            embedding_dim=embedding_dim,
            name="token_position_embedding"
        )(inputs)
        for i in range(num_layers):
            x = keras_nlp.layers.TransformerDecoder(
                intermediate_dim=intermediate_dim,
                num_heads=num_heads,
                dropout=0.1,
                activation="gelu",
                normalize_first=True,
                name=f"decoder_layer_{i+1}"
            )(
                decoder_sequence=x,
                encoder_sequence=None,
            )
        outputs = layers.Dense(vocab_size, name="logits")(x)
        model = keras.Model(inputs=inputs, outputs=outputs, name="char_lm_decoder")
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=0.01),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[PerplexityMetric()],
    )
    return model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

lm_model = create_char_lm_decoder(
    vocab_size=vocab_size
)

early_stopping = EarlyStopping(
    monitor='loss',
    patience=5,
    verbose=1,
    restore_best_weights=True
)

lm_model.fit(
    lm_dataset,
    epochs=100,
    callbacks=[early_stopping]
)

In [ ]:
lm_model.save("char_lm_decoder1.keras")

In [ ]:
from google.colab import files
files.download("char_lm_decoder.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>